# Phase 1 Training — All Experiments

Trains the meta-encoder and per-layer profile regressors for every experiment in one run:

| Key | Config | Description |
|---|---|---|
| `phase1` | `phase1.yaml` | Full model — info + geometry |
| `info_only` | `ablations/info_only.yaml` | Info loss only (λ_geometry = 0) |
| `geometry_only` | `ablations/geometry_only.yaml` | Geometry loss only (info weight = 0) |

**Skips any experiment whose `best.pt` already exists on Drive.**
Only `best.pt` is saved per experiment (no per-epoch checkpoints).

Run this notebook before the evaluation notebooks (nb01–nb07).

## Colab Setup
Clones the repo, installs dependencies, and mounts Google Drive.

In [ ]:
import os, sys

# 1. Clone repository
REPO_DIR = '/content/trainable-circuits'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/jacobposchl/trainable-circuits {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# 2. Install extra dependencies
!pip install hdbscan umap-learn -q

# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 4. Path constants
DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data/raw'   # root for torchvision datasets
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

print('Repo:     ', REPO_DIR)
print('Data:     ', DATA_DIR)
print('Checkpts: ', CKPT_DIR)

In [ ]:
import yaml
import torch
from training.unified_trainer import Phase1Trainer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## Experiments

Define all training runs. Edit this cell to add/remove experiments or change settings.

In [ ]:
EXPERIMENTS = {
    'phase1': {
        'config':   CONFIG_DIR + '/phase1.yaml',
        'ckpt_dir': CKPT_DIR   + '/phase1',
        'desc':     'Full model — info + geometry',
    },
    'info_only': {
        'config':   CONFIG_DIR + '/ablations/info_only.yaml',
        'ckpt_dir': CKPT_DIR   + '/ablations/info_only',
        'desc':     'Ablation: info loss only (λ_geometry = 0)',
    },
    'geometry_only': {
        'config':   CONFIG_DIR + '/ablations/geometry_only.yaml',
        'ckpt_dir': CKPT_DIR   + '/ablations/geometry_only',
        'desc':     'Ablation: geometry loss only (info weight = 0)',
    },
}

for name, exp in EXPERIMENTS.items():
    status = 'DONE' if os.path.exists(exp['ckpt_dir'] + '/best.pt') else 'pending'
    print(f"  [{status:7s}] {name:15s} — {exp['desc']}")

## Train All

Runs each experiment in order. Skips any that already have a `best.pt` on Drive.
Only the best checkpoint is saved — no per-epoch files.

In [ ]:
# Architecture sanity check: confirm the flow-compression backbone is active.
# Expected after the refinement:
#   - layer_dims = [256]*8  (D_flow=256 for all layers)
#   - backbone has traj_compressors and flow_compressors
#   - _flow_targets is populated on forward pass and has shape [B, 256]
from models.backbone import FrozenBackbone
import torch as _torch

_bb = FrozenBackbone(arch='resnet18', num_classes=10, pretrained=False)

assert hasattr(_bb, 'traj_compressors'), \
    "traj_compressors not found — backbone.py may not be updated"
assert hasattr(_bb, 'flow_compressors'), \
    "flow_compressors not found — backbone.py may not be updated"

_dim0 = _bb.layer_dims[0]
assert _dim0 == 256, (
    f"backbone.layer_dims[0] = {_dim0}. Expected 256 (D_flow). "
    "Check models/backbone.py — flow compression may not be active."
)
assert len(_bb.layer_dims) == 8, \
    f"Expected 8 layers, got {len(_bb.layer_dims)}"

# Verify flow targets are populated and shaped correctly
_dummy = _torch.zeros(2, 3, 32, 32)
_ = _bb(_dummy)
assert len(_bb._flow_targets) == 8, \
    f"Expected 8 flow targets after forward, got {len(_bb._flow_targets)}"
assert _bb._flow_targets[0].shape == (2, 256), \
    f"Expected flow target shape (2, 256), got {_bb._flow_targets[0].shape}"

print(f"Architecture check passed:")
print(f"  layer_dims      = {_bb.layer_dims}")
print(f"  flow_target[0]  = {_bb._flow_targets[0].shape}  (D_flow=256)")
del _bb, _dummy

In [ ]:
for name, exp in EXPERIMENTS.items():
    best_path = exp['ckpt_dir'] + '/best.pt'

    if os.path.exists(best_path):
        # Architecture check: verify the checkpoint's projector input dims match
        # the current backbone's layer_dims (flatten vs old GAP-pooled dims differ).
        ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
        ckpt_cfg = ckpt.get('config', {})
        ckpt_arch = ckpt_cfg.get('model', {}).get('arch', 'unknown')

        with open(exp['config']) as _f:
            _cfg = yaml.safe_load(_f)
        _bb = FrozenBackbone(
            arch=_cfg['model']['arch'],
            num_classes=_cfg['model']['num_classes'],
            pretrained=False,   # no download needed, just for layer_dims
        )
        expected_dim0 = _bb.layer_dims[0]
        ckpt_dim0 = ckpt['meta_encoder_state']['projectors.0.0.weight'].shape[1]

        if ckpt_dim0 != expected_dim0:
            print(f"[{name}] Stale checkpoint (projector input dim {ckpt_dim0} != "
                  f"expected {expected_dim0}). Deleting and retraining.")
            os.remove(best_path)
        else:
            metrics = ckpt.get('val_metrics', {})
            print(f"[{name}] Already trained (epoch {ckpt['epoch']+1}, "
                  f"R²={metrics.get('r2', float('nan')):.3f}, "
                  f"ρ={metrics.get('mean_rho', float('nan')):.3f}) — skipping.")
            continue

    print(f'\n{"="*60}')
    print(f'Training: {name}')
    print(f'  {exp["desc"]}')
    print(f'{"="*60}')

    with open(exp['config']) as f:
        config = yaml.safe_load(f)

    config['data']['data_dir']          = DATA_DIR
    config['data']['num_workers']       = 2
    config['logging']['checkpoint_dir'] = exp['ckpt_dir']
    config['logging']['save_every']     = 9999   # best.pt only

    trainer = Phase1Trainer(config)
    trainer.train()
    print(f'[{name}] Training complete.')

In [ ]:
# ── Resume a specific experiment after Colab timeout ─────────────────────────
# If a run was interrupted, you can resume from its best.pt (the last saved
# checkpoint). Re-run setup cells first, then uncomment and run this cell.
#
# resume_name = 'phase1'   # key from EXPERIMENTS dict
# exp         = EXPERIMENTS[resume_name]
# best_path   = exp['ckpt_dir'] + '/best.pt'
#
# with open(exp['config']) as f:
#     config = yaml.safe_load(f)
# config['data']['data_dir']          = DATA_DIR
# config['data']['num_workers']       = 2
# config['logging']['checkpoint_dir'] = exp['ckpt_dir']
# config['logging']['save_every']     = 9999
#
# trainer = Phase1Trainer(config)
# trainer.train(resume_from=best_path)

## Summary

Prints final val metrics for every completed experiment.

In [ ]:
print(f"{'Experiment':<16} {'Epoch':>5}  {'R²':>6}  {'mean ρ':>7}  Status")
print("-" * 50)
for name, exp in EXPERIMENTS.items():
    best_path = exp['ckpt_dir'] + '/best.pt'
    if not os.path.exists(best_path):
        print(f"{name:<16} {'—':>5}  {'—':>6}  {'—':>7}  not trained")
        continue
    ckpt    = torch.load(best_path, map_location='cpu', weights_only=False)
    metrics = ckpt.get('val_metrics', {})
    r2      = metrics.get('r2', float('nan'))
    rho     = metrics.get('mean_rho', float('nan'))
    epoch   = ckpt.get('epoch', -1) + 1
    ok      = 'PASS' if r2 >= 0.7 and rho >= 0.65 else 'FAIL'
    print(f"{name:<16} {epoch:>5}  {r2:>6.3f}  {rho:>7.3f}  {ok}")